# quaero_tree

Decision-tree index + distribution-matching subset selection + lossless reduction.

## Pipeline

**Stage 1 — Index construction.**  Build a decision tree on the *descriptor
variables only*, predicting the **bin id** as a categorical class, grown until
every leaf is pure (single bin).  Built once, reused across queries.  Per node
we store its per-bin count vector, tree topology, per-leaf bin/rows, and the
conjunctive path predicate.

**Stage 2 — Selection.**  Given a target distribution over bins (absolute
counts `n_b`, or proportions `p_b` with a free scaling factor), select a subset
of leaves whose per-bin counts match the target within tolerance (TV distance),
minimising the **reduced tree size** of the induced 0/1 labelling.

Because leaves are pure, each leaf contributes a *scalar* to exactly one bin, so
the bins are independent *in the constraint*; they are coupled *only* through
reduction (a parent collapses only if all descendants are selected, and those
descendants may carry different bin labels).  So we do **not** solve per-bin and
then reduce — we parameterise the search over **internal nodes**: "take this
whole subtree" contributes the node's full per-bin count vector at description
cost 1.  The implemented selector is a **subtree-first greedy** (big fitting
subtrees for bulk mass, then individual leaves patching per-bin residuals) --
matching the expected solution shape.  It is not provably minimal; the exact
objective (minimum antichain of all-selected subtrees under a TV constraint) is
a MILP.

**Stage 3 — Lossless reduction.**  Label every leaf 1/0, walk bottom-up, and
collapse a subtree **only when all of its descendants share a label**.  Never
collapse on a majority.  The reduced tree denotes exactly the selected rows
(same extension, fewer nodes); its all-selected subtree roots are the DNF
conjuncts.  A hard assertion verifies `leaves(A) == selected leaves`.

In [ ]:
import pandas as pd, numpy as np, math
from scipy import stats as scipy_stats
from sklearn.tree import DecisionTreeClassifier
from typing import Callable, Dict, List, Tuple

In [ ]:
def bin_target(df, target_col):
    """Discretise the target with Scott's rule; return (bin id per row, n_bins)."""
    x = df[target_col].dropna().values
    std = np.std(x, ddof=1)
    h = 3.5 * std * (len(x) ** (-1 / 3))
    n_bins = max(1, math.ceil((x.max() - x.min()) / h))
    ids = pd.cut(df[target_col], bins=n_bins, labels=range(n_bins),
                 include_lowest=True).astype(int).to_numpy()
    return ids, n_bins


def _bin_centers(df, target_col, n_bins):
    lo, hi = df[target_col].min(), df[target_col].max()
    e = np.linspace(lo, hi, n_bins + 1)
    return (e[:-1] + e[1:]) / 2


def make_pdf(spec):
    """Single distribution ("gaussian", {"loc":0,"scale":1}) or a weighted mixture."""
    DIST = {"gaussian": scipy_stats.norm, "normal": scipy_stats.norm,
            "exponential": scipy_stats.expon, "beta": scipy_stats.beta}
    if isinstance(spec, tuple):
        name, p = spec; d = DIST[name.lower()]
        return lambda x: d.pdf(x, **p)
    tot = sum(w for *_, w in spec)
    comp = [(DIST[n.lower()], p, w / tot) for n, p, w in spec]
    return lambda x: sum(w * d.pdf(x, **p) for d, p, w in comp)

In [ ]:
def build_index(df, target_col, desc_cols=None):
    """Stage 1: decision tree on descriptors -> bin id, grown to purity.

    Returns the index dict: per-node bin-count vectors, tree topology, per-leaf
    bin/rows, and a path-predicate accessor.  Built once, reused across queries.
    """
    if desc_cols is None:
        desc_cols = [c for c in df.columns if c != target_col]
    y, n_bins = bin_target(df, target_col)
    X = df[desc_cols].to_numpy()

    clf = DecisionTreeClassifier(criterion="gini", min_samples_leaf=1,
                                 min_samples_split=2, random_state=0)
    clf.fit(X, y)                              # grows until pure (or features tie)
    t, C = clf.tree_, clf.classes_
    n = t.node_count
    cl, cr, feat, thr = t.children_left, t.children_right, t.feature, t.threshold
    is_leaf = cl == -1

    parent = np.full(n, -1, int); depth = np.zeros(n, int); st = [(0, 0)]
    while st:
        v, d = st.pop(); depth[v] = d
        if not is_leaf[v]:
            parent[cl[v]] = parent[cr[v]] = v
            st += [(cl[v], d + 1), (cr[v], d + 1)]

    leaves = [v for v in range(n) if is_leaf[v]]
    leaf_of_row = clf.apply(X)
    rows_of_leaf = {}
    for i, lf in enumerate(leaf_of_row):
        rows_of_leaf.setdefault(int(lf), []).append(i)

    # per-node per-bin counts computed from the DATA (robust: tree_.value is
    # normalised to fractions in recent sklearn, so it can't be rounded to counts)
    node_cnt = {v: {} for v in range(n)}
    for lf in leaves:                          # leaf bin-count vectors from y
        c = {}
        for i in rows_of_leaf.get(lf, []):
            b = int(y[i]); c[b] = c.get(b, 0) + 1
        node_cnt[lf] = c
    for v in sorted(range(n), key=lambda v: -depth[v]):   # propagate up (children first)
        if not is_leaf[v]:
            m = {}
            for ch in (cl[v], cr[v]):
                for b, k in node_cnt[ch].items():
                    m[b] = m.get(b, 0) + k
            node_cnt[v] = m

    leaf_bin = {v: max(node_cnt[v], key=node_cnt[v].get) for v in leaves}   # pure -> single
    impure = [v for v in leaves if len(node_cnt[v]) > 1]
    if impure:
        print(f"WARNING: {len(impure)} impure leaves (identical descriptors, different bins); "
              f"leaf_bin uses the majority.")

    def path_predicate(v):
        preds, cur = [], v
        while parent[cur] != -1:
            p = parent[cur]
            op = "<=" if cl[p] == cur else ">"
            preds.append((desc_cols[feat[p]], op, float(thr[p]))); cur = p
        return list(reversed(preds))

    return dict(clf=clf, desc_cols=desc_cols, n_bins=n_bins, n=n, is_leaf=is_leaf,
                cl=cl, cr=cr, parent=parent, depth=depth, node_cnt=node_cnt,
                leaves=leaves, leaf_bin=leaf_bin, rows_of_leaf=rows_of_leaf,
                path_predicate=path_predicate)

In [ ]:
def leaves_under(ix, v):
    """All leaf ids under node v."""
    out, st = [], [v]
    while st:
        u = st.pop()
        (out.append(u) if ix["is_leaf"][u] else st.extend([ix["cl"][u], ix["cr"][u]]))
    return out


def select_subtrees(ix, target, total=None, cap_slack=0.02):
    """Stage 2: subtree-first greedy selection.

    target : {bin_id: proportion} (preferred; free scale via `total`) or absolute counts.
    total  : desired total selected (for proportions).  If None with proportions,
             uses the largest feasible scale.

    Takes the largest whole subtrees whose full count-vector fits under per-bin
    caps -- this is where the tree coupling is exploited (one description-cost per
    subtree) -- then patches per-bin residuals with individual leaves.

    NOTE: greedy, not provably minimal.  The exact objective (minimum antichain of
    all-selected subtrees under a TV constraint) is a MILP.
    """
    bins = sorted(ix["node_cnt"][0])                      # bins present at the root
    s = sum(target.values())
    if total is None and s <= 1.5:                        # proportions -> max feasible scale
        total = min(ix["node_cnt"][0].get(b, 0) / target[b] for b in bins if target.get(b, 0) > 0)
    tc = {b: target.get(b, 0) * (total if s <= 1.5 else 1) for b in bins}   # target counts
    cap = {b: tc[b] * (1 + cap_slack) for b in bins}

    selected, got = set(), {b: 0.0 for b in bins}
    internal = [v for v in range(ix["n"]) if not ix["is_leaf"][v]]
    tot = lambda v: sum(ix["node_cnt"][v].values())
    for v in sorted(internal, key=lambda v: (ix["depth"][v], -tot(v))):     # big & high first
        lu = leaves_under(ix, v)
        if any(l in selected for l in lu):
            continue
        c = ix["node_cnt"][v]
        if all(got[b] + c.get(b, 0) <= cap[b] + 1e-9 for b in bins):
            selected.update(lu)
            for b, k in c.items():
                got[b] += k
    for lf in sorted(ix["leaves"], key=lambda l: -ix["node_cnt"][l].get(ix["leaf_bin"][l], 0)):
        if lf in selected:
            continue
        b = ix["leaf_bin"][lf]; k = ix["node_cnt"][lf].get(b, 0)
        if got[b] + k <= cap[b] + 1e-9 and got[b] < tc[b] - 1e-9:
            selected.add(lf); got[b] += k

    labels = {lf: int(lf in selected) for lf in ix["leaves"]}
    return labels, got, tc


def tv_distance(got, tc):
    """Total-variation distance between the achieved and target bin histograms."""
    gs, ts = sum(got.values()), sum(tc.values())
    if gs == 0 or ts == 0:
        return 1.0
    bins = set(got) | set(tc)
    return 0.5 * sum(abs(got.get(b, 0) / gs - tc.get(b, 0) / ts) for b in bins)

In [ ]:
def reduce_labeling(ix, labels):
    """Stage 3: lossless bottom-up reduction.

    Collapse a subtree ONLY when ALL its descendants share a label.  Returns the
    maximal all-selected subtree roots (the DNF conjuncts), the reduced tree size,
    and the DNF predicate.  A hard assertion verifies the reduction is lossless
    (leaves(A) == selected leaves).
    """
    n, parent = ix["n"], ix["parent"]
    lab = np.full(n, -1, int)                             # -1 mixed, else the shared label
    for v in sorted(range(n), key=lambda v: -ix["depth"][v]):   # children before parents
        if ix["is_leaf"][v]:
            lab[v] = labels[v]
        else:
            a, b = lab[ix["cl"][v]], lab[ix["cr"][v]]
            lab[v] = a if (a == b and a != -1) else -1

    A = [v for v in range(n) if lab[v] == 1 and (parent[v] == -1 or lab[parent[v]] != 1)]
    reduced = [v for v in range(n) if parent[v] == -1 or lab[parent[v]] == -1]

    # lossless check: the reduced labelling denotes exactly the selected leaves
    covered = set().union(*[set(leaves_under(ix, v)) for v in A]) if A else set()
    sel = {lf for lf in ix["leaves"] if labels[lf] == 1}
    assert covered == sel, "reduction changed the extension -- not lossless!"

    dnf = [ix["path_predicate"](v) for v in A]            # OR of conjunctive predicates
    return dict(node_label=lab, antichain=A, dnf=dnf,
                reduced_size=len(reduced), n_terms=len(A))


def dnf_to_str(dnf, max_terms=None):
    """Human-readable DNF: OR of ANDed (feature op threshold) predicates."""
    terms = dnf if max_terms is None else dnf[:max_terms]
    lines = []
    for term in terms:
        if term:
            lines.append("(" + " AND ".join(f"{f} {op} {t:.3f}" for f, op, t in term) + ")")
        else:
            lines.append("(TRUE)")                        # root subtree = everything
    tail = "" if (max_terms is None or len(dnf) <= max_terms) else f"\n  ... (+{len(dnf)-max_terms} more)"
    return "  " + "\n  OR ".join(lines) + tail

In [ ]:
# --- demo: Gaussian blob dataset with a planted subgroup --------------------
np.random.seed(42); rng = np.random.default_rng(42)
N        = 1500
n_blobs  = 4
blob_std = 0.08

# descriptors: random Gaussian blobs (clustered, non-uniform)
blob_centers = rng.uniform(0.0, 1.0, size=(n_blobs, 3))
assign       = rng.integers(0, n_blobs, size=N)
coords       = blob_centers[assign] + rng.normal(0.0, blob_std, size=(N, 3))
df = pd.DataFrame(coords, columns=["feature_a", "feature_b", "feature_c"])
df["target"] = rng.random(N)                              # background target ~ Uniform[0, 1)

# plant a subgroup: one whole blob gets target ~ N(0.5, 0.15) (the specified dist.)
planted_blob = 0
condition = pd.Series(assign == planted_blob)
df.loc[condition, "target"] = rng.normal(loc=0.5, scale=0.15, size=int(condition.sum()))
print(f"planted rows (blob {planted_blob}): {int(condition.sum())} / {N}")

# Stage 1: build the index once
ix = build_index(df, "target")
print(f"tree index: {ix['n']} nodes, {len(ix['leaves'])} leaves, {ix['n_bins']} bins")

# target distribution over bins, from the specified pdf
pdf = make_pdf(("gaussian", {"loc": 0.5, "scale": 0.15}))
bin_ctr = _bin_centers(df, "target", ix["n_bins"])
p = np.array([pdf(bin_ctr[b]) for b in range(ix["n_bins"])]); p = p / p.sum()
target = {b: float(p[b]) for b in range(ix["n_bins"])}

# Stage 2: select leaves matching the target at ~30% coverage
labels, got, tc = select_subtrees(ix, target, total=round(0.30 * N))

# Stage 3: lossless reduction -> DNF
red = reduce_labeling(ix, labels)

print(f"\nselected leaves : {sum(labels.values())} / {len(labels)}")
print(f"selected rows   : {sum(k for k in got.values()):.0f}")
print(f"TV distance     : {tv_distance(got, tc):.4f}")
print(f"reduced size    : {red['reduced_size']} nodes")
print(f"DNF terms       : {red['n_terms']}  (subtrees carrying bulk mass + leaf patches)")
print("\nDNF (first 5 conjuncts):")
print(dnf_to_str(red["dnf"], max_terms=5))

In [ ]:
# Per-bin achieved vs target counts
rows = [{"bin": b, "target": round(tc.get(b, 0), 1), "achieved": round(got.get(b, 0), 1)}
        for b in sorted(tc)]
report = pd.DataFrame(rows)
report.loc["TOTAL"] = ["TOTAL", round(sum(tc.values()), 1), round(sum(got.values()), 1)]
display(report)

# subtree-size profile of the DNF terms: how many rows each conjunct covers
sizes = sorted((sum(ix["node_cnt"][v].values()) for v in red["antichain"]), reverse=True)
print(f"DNF term coverage (rows per conjunct), largest first: {sizes[:15]}"
      f"{' ...' if len(sizes) > 15 else ''}")
print(f"bulk (largest term) covers {sizes[0] if sizes else 0} rows; "
      f"{sum(1 for s in sizes if s == 1)} single-leaf patches")

In [ ]:
import matplotlib.pyplot as plt

def _split_label(ix, v):
    t = ix["clf"].tree_
    return f"{ix['desc_cols'][t.feature[v]]}\n<= {t.threshold[v]:.2f}"

def _draw_tree(ix, children_fn, color_fn, label_fn, max_depth=4, title="", ax=None):
    """Generic top-down tree renderer.  children_fn(v) -> list of drawn children
    ([] makes v a terminal); nodes past max_depth become summary '...' terminals."""
    pos, kind, xc = {}, {}, [0]

    def eff(v, d):
        return [] if d >= max_depth else children_fn(v)

    def rec(v, d):
        ch = eff(v, d)
        if not ch:
            x = xc[0]; xc[0] += 1
            kind[v] = "leaf" if not children_fn(v) else "cut"
        else:
            xs = [rec(c, d + 1) for c in ch]
            x = sum(xs) / len(xs); kind[v] = "internal"
        pos[v] = (x, -d)
        return x

    rec(0, 0)
    W = max(1, xc[0])
    if ax is None:
        _, ax = plt.subplots(figsize=(min(26, 1.2 * W + 2), 1.5 + max_depth * 1.4))
    for v, (x, y) in pos.items():
        for c in eff(v, int(-y)):
            if c in pos:
                cx, cy = pos[c]
                ax.plot([x, cx], [y, cy], color="0.6", lw=0.8, zorder=1)
    for v, (x, y) in pos.items():
        ax.text(x, y, label_fn(v, kind[v]), ha="center", va="center", fontsize=7, zorder=2,
                bbox=dict(boxstyle="round,pad=0.3", fc=color_fn(v), ec="0.4", lw=0.6))
    ax.set_title(title, fontsize=11); ax.axis("off"); ax.margins(0.06)
    return ax


def draw_index_tree(ix, max_depth=4, ax=None):
    """Stage 1 result: the full decision-tree index (leaves coloured by bin)."""
    cmap = plt.get_cmap("tab20")
    children = lambda v: [] if ix["is_leaf"][v] else [ix["cl"][v], ix["cr"][v]]
    color = lambda v: (cmap(ix["leaf_bin"][v] % 20) if ix["is_leaf"][v] else "0.85")
    def label(v, kind):
        if kind == "internal":
            return _split_label(ix, v)
        rows = sum(ix["node_cnt"][v].values())
        return f"... {rows}" if kind == "cut" else f"bin {ix['leaf_bin'][v]}\n{rows}"
    return _draw_tree(ix, children, color, label, max_depth, "Stage 1 - index tree", ax)


_LAB_COLOR = {1: "#66bb6a", 0: "#eeeeee", -1: "#cfd8dc"}    # selected / unselected / mixed

def draw_labeled_tree(ix, red, max_depth=5, ax=None):
    """Before simplification: the full labelled tree (leaves 1/0, internal nodes
    coloured by whether their whole subtree agrees)."""
    lab = red["node_label"]
    children = lambda v: [] if ix["is_leaf"][v] else [ix["cl"][v], ix["cr"][v]]
    color = lambda v: _LAB_COLOR[int(lab[v])]
    def label(v, kind):
        if kind == "internal":
            return _split_label(ix, v)
        rows = sum(ix["node_cnt"][v].values())
        tag = {1: "SEL", 0: "-", -1: "mix"}[int(lab[v])]
        return f"... {rows}" if kind == "cut" else f"{tag}\n{rows}"
    return _draw_tree(ix, children, color, label, max_depth,
                      "Before simplification (labelled tree)", ax)


def draw_reduced_tree(ix, red, max_depth=6, ax=None):
    """After simplification: children stop at any node whose whole subtree agrees,
    so each maximal all-selected subtree collapses to one green terminal (a DNF
    conjunct)."""
    lab = red["node_label"]
    def children(v):
        return [] if (ix["is_leaf"][v] or lab[v] != -1) else [ix["cl"][v], ix["cr"][v]]
    color = lambda v: _LAB_COLOR[int(lab[v])]
    def label(v, kind):
        if kind == "internal":
            return _split_label(ix, v)
        rows = sum(ix["node_cnt"][v].values())
        tag = {1: "SEL", 0: "-", -1: "mix"}[int(lab[v])]
        return f"... {rows}" if kind == "cut" else f"{tag}\n{rows}"
    return _draw_tree(ix, children, color, label, max_depth,
                      "After simplification (reduced tree)", ax)

In [ ]:
# Small illustrative example so the selection and collapse are actually visible
# (the main demo tree has thousands of leaves).  Descriptors here PREDICT the bin,
# so the tree is shallow.
rng2 = np.random.default_rng(1)
Nt = 600
a = rng2.uniform(0, 1, Nt); b = rng2.uniform(0, 1, Nt)
region = np.floor(a * 3) + 3 * np.floor(b * 3)          # 9 descriptor regions
dft = pd.DataFrame({"a": a, "b": b, "target": region * 8 + 10 + rng2.normal(0, 1.5, Nt)})

ixt = build_index(dft, "target")
print(f"toy index: {ixt['n']} nodes, {len(ixt['leaves'])} leaves, {ixt['n_bins']} bins")

# target = a contiguous band of bins, in their natural proportions, so it is
# cleanly achievable by taking whole regions (each collapses to one DNF term)
bins_t = sorted(ixt["node_cnt"][0])
avail  = {bb: ixt["node_cnt"][0][bb] for bb in bins_t}
chosen = bins_t[2:7]                                     # a middle band of bins
tgt    = {bb: (avail[bb] if bb in chosen else 0.0) for bb in bins_t}
Z = sum(tgt.values()); tgt = {bb: tgt[bb] / Z for bb in bins_t}

labels_t, got_t, tc_t = select_subtrees(ixt, tgt, total=int(sum(avail[bb] for bb in chosen)))
red_t = reduce_labeling(ixt, labels_t)
print(f"selected leaves: {sum(labels_t.values())}/{len(labels_t)}  "
      f"TV={tv_distance(got_t, tc_t):.3f}  "
      f"reduced_size={red_t['reduced_size']}  DNF_terms={red_t['n_terms']}")

In [ ]:
# Draw the three trees
draw_index_tree(ixt, max_depth=4)
plt.tight_layout(); plt.show()

draw_labeled_tree(ixt, red_t, max_depth=5)
plt.tight_layout(); plt.show()

draw_reduced_tree(ixt, red_t, max_depth=6)
plt.tight_layout(); plt.show()